#**TP 1 - Modelagem Preditiva e Construção de Agentes de IA**

## Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Preparação dos dados

In [2]:
rng = np.random.default_rng(7)
n = 12000

plano = rng.choice(["basico", "padrao", "premium"], size=n, p=[0.4, 0.4, 0.2])
metodo_pagamento = rng.choice(["cartao_credito", "boleto", "pix"], size=n, p=[0.55, 0.20, 0.25])
dispositivo_principal = rng.choice(["smart_tv", "celular", "computador", "tablet"], size=n, p=[0.35, 0.35, 0.20, 0.10])
tempo_assinatura_meses = np.round(np.clip(rng.exponential(scale=14, size=n), 0, 72), 1)
uso_medio_semanal_horas = np.round(np.clip(rng.normal(9, 4, n), 0, 40), 1)
numero_reclamacoes = rng.poisson(0.4, n)

# churn: menor tempo de assinatura, menor uso e mais reclamações aumentam a chance de cancelar
logit_churn = (
    -0.85
    - 0.04 * tempo_assinatura_meses
    - 0.06 * uso_medio_semanal_horas
    + 0.5 * numero_reclamacoes
    + rng.normal(0, 0.6, n)
)
prob_churn = 1 / (1 + np.exp(-logit_churn))
churn = (rng.uniform(0, 1, n) < prob_churn).astype(int)

# ltv_alto: assinantes de alto valor formam uma região (não um corte linear simples) no espaço
# tempo_assinatura x uso_medio_semanal, com alguma sobreposição de classes nas bordas
raio = np.sqrt(((tempo_assinatura_meses - 40) / 20) ** 2 + ((uso_medio_semanal_horas - 14) / 8) ** 2)
ltv_alto = (raio < 1.0).astype(int)
inverter = rng.random(n) < 0.05
ltv_alto = np.where(inverter, 1 - ltv_alto, ltv_alto)

df = pd.DataFrame({
    "cliente_id": [f"AST-{i:05d}" for i in range(1, n + 1)],
    "plano": plano,
    "metodo_pagamento": metodo_pagamento,
    "dispositivo_principal": dispositivo_principal,
    "tempo_assinatura_meses": tempo_assinatura_meses,
    "uso_medio_semanal_horas": uso_medio_semanal_horas,
    "numero_reclamacoes": numero_reclamacoes,
    "churn": churn,
    "ltv_alto": ltv_alto,
})

df.to_csv("assinantes_neostream.csv", index=False)

## **EXERCÍCIO 1**

### 1.1 Para cada um dos quatro problemas, classifique o tipo de aprendizado (supervisionado ou não supervisionado) e justifique com base na presença ou ausência de rótulos no problema descrito.

- **Previsão de cancelamento:** Aprendizado supervisionado. Já que existe um histórico dos assinantes que cancelaram ou não, e esse resultado funciona como um rótulo para treinar um classificador.
- **Agrupamento por padrão de consumo:** Aprendizado não supervisionado. Como não há rótulos prévios, o algoritmo precisa descobrir os grupos sozinho.
- **Recomendação de séries em tempo real:** Aprendizado não supervisionado. Não existe uma categoria correta a ser prevista, a recomendação deve se basear em uma similaridade entre assinantes e séries a partir dos padrões de consumo, sem nenhum rótulo explícito.
- **Ajuste de preço do plano promocional:** Aprendizado supervisionado. Cada reação do mercado, como a compra ou não ou variação de demanda, fornece um sinal associado a cada preço testado que pode ser observável.

### 1.2 Para cada problema, classifique o modo de treinamento (batch ou online) considerando a frequência de chegada de novos dados e a necessidade de resposta em tempo real.

- **Previsão de cancelamento:** Batch. Já que a previsão é mensal e o histórico pode ser reprocessado em lote, e não existe uma urgência de uma resposta imediata.
- **Agrupamento por padrão de consumo:** Batch. A segmentação é uma análise periódica que ocorre sobre toda a base, sem a exigência de uma atualização instantânea.
- **Recomendação de séries em tempo real:** Online. Em um streaming, os conteúdos disponíveis para serem assistidos mudam constântemente, e a tendência do que um assinante pode dar preferência a assistir é momentâneo, o que exige atualização constante.
- **Ajuste de preço do plano promocional:** Online. O preço precisa ser ajustado conforme cada nova reação do mercado, isso faz com que a atualização incremental do modelo a cada novo dado seja uma exigência.

### 1.3 Produza uma tabela problema | tipo_de_aprendizado | modo_de_treinamento | justificativa em um bloco de texto no notebook.

| problema | tipo_de_aprendizado | modo_de_treinamento | justificativa |
|---|---|---|---|
| Prever quais assinantes vão cancelar no próximo mês | Supervisionado | Batch | Há rótulo histórico (cancelou ou não); A previsão é mensal e não exige resposta em tempo real. |
| Agrupar assinantes por padrão de consumo | Não supervisionado | Batch | Não há rótulos prévios; A segmentação é periódica sobre toda a base. |
| Recomendar séries em tempo real conforme o assinante assiste | Não supervisionado | Online | Não há rótulo explícito, apenas similaridade entre padrões de consumo; Novos eventos chegam continuamente e exigem resposta imediata. |
| Ajustar o preço de um plano promocional a cada reação do mercado | Supervisionado | Online | Cada reação do mercado fornece um sinal observável para o preço testado; O ajuste ocorre a cada novo dado. |

## **EXERCÍCIO 2**

### 2.1 Diagnostique cada protótipo em termos de bias e variância, explicando o que a diferença entre acerto de treino e de teste revela sobre cada um.

**Protótipo 1 (98% treino / 61% teste):** a diferença de 37 pontos percentuais indica uma alta variância e um baixo bias. O modelo ajusta quase perfeitamente os dados de treino, o que caracteriza um baixo bias, mas decorou ruídos e particularidades da amostra e não generaliza, já que no teste cai para 61%, apenas 11 pontos acima de um baseline de 50%. Tudo isso caracteriza um caso típico de overfitting.

**Protótipo 2 (71% treino / 69% teste):** a diferença de apenas 2 pontos indica baixa variância, ou seja, o modelo possuí um comportamento forma estável em dados vistos e não vistos. Porém, a acurácia é não é muito boa mesmo no treino, o que sugere um bias relativamente alto, u seja, o modelo é simples demais para capturar todo o padrão disponível. É um caso de underfitting leve.


### 2.2 Explique, em termos do tradeoff bias-variância, por que aumentar a regularização ajudaria um protótipo e prejudicaria o outro.

A regularização traz uma penalidade a complexidade do modelo, o que aumenta o bias, porém reduz a variância. Por isso o mesmo ajuste tem efeitos opostos nos dois protótipos:

- No Protótipo 1, o problema dominante é a variância. Uma maior regularização restringe o modelo, impedindo que ele memorize ruído e reduzindo a distância entre treino e teste. O bias sobe um pouco, ou seja, a acurácia de treino cai, mas a acurácia de teste tende a melhorar.
- No Protótipo 2, o problema dominante já é o bias. Mais regularização deixaria ele ainda mais restrito, piorando um ajuste que já é limitado e reduzindo a acurácia tanto no treino quanto no teste, sem ter nenhum ganho de generalização, já que a variância já é baixa.

### 2.3 Recomende por escrito qual dos dois protótipos deve receber mais regularização e qual deve receber menos, com a justificativa técnica completa.

- **Protótipo 1:** aumentar a regularização. A lacuna de 37 pontos entre a acurácia de treino e teste mostra que o erro vem da variância, que é justamente o que a regularização reduz. Isso faz com que tenha uma queda no treino, porém um ganho no teste, que é a métrica que realmente importa para produção.
- **Protótipo 2:** reduzir a regularização. Com variância já baixa, há margem para dar mais flexibilidade ao modelo e diminuir o bias. Como a acurácia de teste já está bem acima do baseline, o caminho é aliviar a restrição, possivelmente combinada com mais features ou um modelo mais expressivo.

Como o objetivo é um modelo que generalize, a decisão se baseia na acurácia de teste e na diferença treino-teste, e não na acurácia de treino isoladamente. Nesse critério, o Protótipo 2 já é hoje o mais confiável, e o Protótipo 1 precisa de correção por regularização antes de ser considerado.

## **EXERCÍCIO 3**

### 3.1 Carregue assinantes_neostream.csv com pandas e separe a coluna churn como alvo.

In [3]:
df = pd.read_csv("assinantes_neostream.csv")
X, y = df.drop(columns = "churn"), df["churn"]

print(f"X: {X.shape} | y: {y.shape}")
print(list(X.columns))

X: (12000, 8) | y: (12000,)
['cliente_id', 'plano', 'metodo_pagamento', 'dispositivo_principal', 'tempo_assinatura_meses', 'uso_medio_semanal_horas', 'numero_reclamacoes', 'ltv_alto']


O arquivo `assinantes_neostream.csv` foi carregado com pandas e dividido em duas partes,  `X`, com as colunas usadas como entrada do modelo, e `y`, com a coluna `churn`.

### 3.2 Use train_test_split com stratify na coluna churn e test_size=0.2 para gerar os conjuntos X_train, X_test, y_train, y_test.

In [4]:
TAMANHO_TESTE = 0.2
SEMENTE = 42


def dividir_treino_teste(X, y):
    return train_test_split(
        X, y, test_size = TAMANHO_TESTE, stratify = y, random_state = SEMENTE
    )


X_train, X_test, y_train, y_test = dividir_treino_teste(X, y)

print(f"Treino: {X_train.shape[0]} linhas | Teste: {X_test.shape[0]} linhas")

Treino: 9600 linhas | Teste: 2400 linhas


A divisão realizada usa `test_size = 0.2`, o que faz com que seja 80% para treino e 20% para teste, e `stratify = y`, que faz o `train_test_split` manter a proporção de cada classe de `churn` nos dois conjuntos.
Isso é importante porque o churn é apenas 18% da base, sem estratificação, o sorteio poderia gerar um conjunto de teste com mais ou menos cancelamentos que o treino, distorcendo a avaliação. O `random_state` foi fixado para que a divisão seja igual e reproduzível toda vez que o notebook for rodado.

### 3.3 Comprove, com value_counts(normalize=True), que a proporção de churn é aproximadamente igual nos dois conjuntos e registre a comparação num bloco de texto.

In [5]:
comparacao_churn = pd.DataFrame({
    "original": y.value_counts(normalize = True),
    "treino": y_train.value_counts(normalize = True),
    "teste": y_test.value_counts(normalize = True)
})

comparacao_churn.style.format("{:.2%}")

,original,treino,teste
churn,,,
0,82.44%,82.44%,82.46%
1,17.56%,17.56%,17.54%


A proporção de cada classe foi calculada com o `value_counts(normalize = True)`, mostrando que 17,54% da base teve churn, e os outros 82,44% não teve, tanto na base original, no treino e no teste.

### 3.4 Explique, em uma frase, por que qualquer fit de scaler ou encoder deve ocorrer somente depois dessa divisão.

Qualquer fit de scaler ou encoder deve ocorrer somente depois da divisão porque ele aprende estatísticas dos dados, como médias, desvios, categorias, e se isso for feito com a base completa, a informação do conjunto de teste vaza para o treino e torna a avaliação otimista.

## **EXERCÍCIO 4**

### 4.1 Construa um ColumnTransformer que aplique StandardScaler às colunas numéricas e OneHotEncoder às colunas categóricas listadas no contexto.

In [6]:
colunas_numericas = ["tempo_assinatura_meses", "uso_medio_semanal_horas", "numero_reclamacoes"]
colunas_categoricas = ["plano", "metodo_pagamento", "dispositivo_principal"]

preprocessador = ColumnTransformer(
    transformers = [
        ("numericas", StandardScaler(), colunas_numericas),
        ("categoricas", OneHotEncoder(handle_unknown = "ignore"), colunas_categoricas),
    ]
)

preprocessador

ColumnTransformer(transformers=[('numericas', StandardScaler(),
                                 ['tempo_assinatura_meses',
                                  'uso_medio_semanal_horas',
                                  'numero_reclamacoes']),
                                ('categoricas',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['plano', 'metodo_pagamento',
                                  'dispositivo_principal'])])

Foi criado um `ColumnTransformer` que aplica os seguintes tratamentos para cada tipo de coluna:

- **Colunas numéricas** (`tempo_assinatura_meses`, `uso_medio_semanal_horas`, `numero_reclamacoes`): `StandardScaler`, padroniza cada coluna para média 0 e desvio padrão 1, deixando todas na mesma escala.
- **Colunas categóricas** (`plano`, `metodo_pagamento`, `dispositivo_principal`): `OneHotEncoder`, transforma cada categoria em uma coluna de 0 ou 1. O `handle_unknown = "ignore"` evita erro caso apareça uma categoria que não existia no treino em dados novos.

As outras colunas que não foram listadas (`cliente_id` e `ltv_alto`) são descartadas pelo `ColumnTransformer`, então não entram no modelo.

### 4.2 Ajuste o ColumnTransformer apenas com X_train (nunca com X_test) e aplique a transformação em ambos os conjuntos.

O `ColumnTransformer` foi ajustado apenas com o `X_train`, usando `fit_transform`. Dessa forma, o `StandardScaler` aprendeu médias e desvios, e o `OneHotEncoder` aprendeu as categorias, somente a partir do treino. Em `X_test` foi usado apenas o `transform`, que aplica o que foi aprendido no treino sem calcular nada novo. Com isso, nenhuma informação do teste influencia o pré-processamento.

### 4.3 Salve o código em preprocessamento_heterogeneo.py e imprima o formato (shape) da matriz resultante para X_train transformado.

In [7]:
%%writefile preprocessamento_heterogeneo.py
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("assinantes_neostream.csv")
X, y = df.drop(columns = "churn"), df["churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, stratify = y, random_state = 42
)

colunas_numericas = ["tempo_assinatura_meses", "uso_medio_semanal_horas", "numero_reclamacoes"]
colunas_categoricas = ["plano", "metodo_pagamento", "dispositivo_principal"]

preprocessador = ColumnTransformer(
    transformers = [
        ("numericas", StandardScaler(), colunas_numericas),
        ("categoricas", OneHotEncoder(handle_unknown = "ignore"), colunas_categoricas),
    ]
)

X_train_transformado = preprocessador.fit_transform(X_train)
X_test_transformado = preprocessador.transform(X_test)

print(f"Formato de X_train transformado: {X_train_transformado.shape}")
print(f"Colunas geradas: {list(preprocessador.get_feature_names_out())}")

Writing preprocessamento_heterogeneo.py


In [ ]:
%run preprocessamento_heterogeneo.py

Formato de X_train transformado: (9600, 13)
Colunas geradas: ['numericas__tempo_assinatura_meses', 'numericas__uso_medio_semanal_horas', 'numericas__numero_reclamacoes', 'categoricas__plano_basico', 'categoricas__plano_padrao', 'categoricas__plano_premium', 'categoricas__metodo_pagamento_boleto', 'categoricas__metodo_pagamento_cartao_credito', 'categoricas__metodo_pagamento_pix', 'categoricas__dispositivo_principal_celular', 'categoricas__dispositivo_principal_computador', 'categoricas__dispositivo_principal_smart_tv', 'categoricas__dispositivo_principal_tablet']


O código foi salvo em `preprocessamento_heterogeneo.py`. O arquivo roda sozinho, carregando o CSV, fazendo a divisão estratificada, ajustando o `ColumnTransformer` com `X_train` e imprimindo o formato.

O `X_train` transformado tem formato (9600, 13): com 9600 linhas (80% das 12 mil) e 13 colunas, sendo 3 numéricas padronizadas e 10 geradas pelo `OneHotEncoder`, 3 de `plano`, 3 de `metodo_pagamento` e 4 de `dispositivo_principal`.

## **EXERCÍCIO 5**

### 5.1 Construa um Pipeline sklearn que combine o ColumnTransformer do Exercício 4 com um LogisticRegression como estimador final.

In [8]:
pipeline = Pipeline(
    steps = [
        ("preprocessamento", preprocessador),
        ("classificador", LogisticRegression()),
    ]
)

pipeline

Pipeline(steps=[('preprocessamento',
                 ColumnTransformer(transformers=[('numericas', StandardScaler(),
                                                  ['tempo_assinatura_meses',
                                                   'uso_medio_semanal_horas',
                                                   'numero_reclamacoes']),
                                                 ('categoricas',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['plano', 'metodo_pagamento',
                                                   'dispositivo_principal'])])),
                ('classificador', LogisticRegression())])

Foi construído um `Pipeline` com duas etapas encadeadas, `preprocessamento`, que é o `ColumnTransformer` do Exercício 4, e `classificador`, um `LogisticRegression` com os hiperparâmetros padrão. Dessa forma, o pré-processamento e o modelo ficam reunidos em um único objeto.

### 5.2 Treine o Pipeline completo com X_train, y_train usando pipeline.fit(...) numa única chamada, sem transformar os dados manualmente antes.

In [9]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessamento',
                 ColumnTransformer(transformers=[('numericas', StandardScaler(),
                                                  ['tempo_assinatura_meses',
                                                   'uso_medio_semanal_horas',
                                                   'numero_reclamacoes']),
                                                 ('categoricas',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['plano', 'metodo_pagamento',
                                                   'dispositivo_principal'])])),
                ('classificador', LogisticRegression())])

O Pipeline foi treinado com uma única chamada, `pipeline.fit(X_train, y_train)`, recebendo o `X_train` bruto, sem nenhuma transformação manual antes. Internamente, ele ajusta o `ColumnTransformer` apenas com o treino, transforma esses dados e treina o `LogisticRegression` com o resultado.

### 5.3 Gere previsões para X_test com pipeline.predict(...) e salve o código em pipeline_classificacao.py.

In [10]:
previsoes = pipeline.predict(X_test)

print(f"Previsões geradas: {previsoes.shape[0]}")
pd.Series(previsoes).value_counts()

Previsões geradas: 2400


,count
0,2391
1,9


As previsões foram geradas com `pipeline.predict(X_test)`, que recebe o `X_test` bruto, o próprio Pipeline aplica o `transform` já ajustado no treino e entrega o resultado ao classificador. Foram geradas 2400 previsões, sendo 2391 de não churn (classe 0) e apenas 9 de churn (classe 1). Como cerca de 17,5% do teste cancelou (aproximadamente 421 assinantes), o modelo quase nunca prevê cancelamento com o limiar padrão.

In [11]:
%%writefile pipeline_classificacao.py
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("assinantes_neostream.csv")
X, y = df.drop(columns = "churn"), df["churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, stratify = y, random_state = 42
)

colunas_numericas = ["tempo_assinatura_meses", "uso_medio_semanal_horas", "numero_reclamacoes"]
colunas_categoricas = ["plano", "metodo_pagamento", "dispositivo_principal"]

preprocessador = ColumnTransformer(
    transformers = [
        ("numericas", StandardScaler(), colunas_numericas),
        ("categoricas", OneHotEncoder(handle_unknown = "ignore"), colunas_categoricas),
    ]
)

pipeline = Pipeline(
    steps = [
        ("preprocessamento", preprocessador),
        ("classificador", LogisticRegression()),
    ]
)

pipeline.fit(X_train, y_train)
previsoes = pipeline.predict(X_test)

print(f"Previsões geradas: {previsoes.shape[0]}")
print(pd.Series(previsoes).value_counts())

Writing pipeline_classificacao.py


In [12]:
%run pipeline_classificacao.py

Previsões geradas: 2400
0    2391
1       9
Name: count, dtype: int64


O código foi salvo em `pipeline_classificacao.py`. O arquivo é independente do notebook, ele carrega os dados, divide, monta e treina o Pipeline e gera as previsões. Ao ser executado, produz o mesmo resultado, 2400 previsões, com 9 de churn.

### 5.4 Explique em duas frases por que empacotar tudo em um Pipeline reduz o risco de data leakage em produção.

Ao empacotar tudo em um Pipeline, o `fit` do scaler e do encoder acontece automaticamente apenas com os dados de treino recebidos, na validação cruzada, apenas com as dobras de treino de cada rodada, então nenhuma informação de teste ou validação contamina o aprendizado. Em produção, o mesmo objeto aplica exatamente as transformações aprendidas no treino a cada novo lote com uma única chamada a `predict`, eliminando passos manuais e o risco de treino e produção usarem transformações diferentes.

## **EXERCÍCIO 6**

### 6.1 Defina uma grade de valores para o hiperparâmetro C do LogisticRegression dentro do Pipeline do Exercício 5 (por exemplo, [0.01, 0.1, 1, 10, 100]).

In [13]:
grade_c = {"classificador__C": [0.01, 0.1, 1, 10, 100]}

O `C` é o inverso da força de regularização do `LogisticRegression`. Valores pequenos regularizam mais e valores grandes regularizam menos. A grade usa de cinco valores em escala logarítmica, o que cobre de regularização forte a muito fraca com poucas combinações, já que orçamento computacional é limitado. Como o `LogisticRegression` está dentro do Pipeline, o nome do parâmetro leva o prefixo do passo `classificador__C`.

### 6.2 Configure GridSearchCV com cv=5 sobre o Pipeline, usando a grade definida, e execute a busca com X_train, y_train.

In [14]:
busca = GridSearchCV(pipeline, param_grid = grade_c, cv = 5, scoring = "roc_auc")
busca.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessamento',
                                        ColumnTransformer(transformers=[('numericas',
                                                                         StandardScaler(),
                                                                         ['tempo_assinatura_meses',
                                                                          'uso_medio_semanal_horas',
                                                                          'numero_reclamacoes']),
                                                                        ('categoricas',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['plano',
                                                                          'metodo_pagamento',
                                                                          'dispositivo_principal'])])),
                                       ('classificador',
                                        LogisticRegression())]),
             param_grid={'classificador__C': [0.01, 0.1, 1, 10, 100]},
             scoring='roc_auc')

O `GridSearchCV` recebeu o Pipeline inteiro, com a grade definida e `cv = 5`, e a busca foi executada apenas com `X_train` e `y_train`, sem tocar no teste.

### 6.3 Reporte o melhor valor de C encontrado (best_params_) e a métrica de validação correspondente (best_score_).

In [15]:
print(f"best_params_: {busca.best_params_}")
print(f"best_score_: {busca.best_score_:.4f}")

resultados_busca = pd.DataFrame(busca.cv_results_)[
    ["param_classificador__C", "mean_test_score", "std_test_score", "rank_test_score"]
].rename(columns = {
    "param_classificador__C": "C",
    "mean_test_score": "auc_medio",
    "std_test_score": "desvio_padrao",
    "rank_test_score": "ranking",
})

resultados_busca.style.format({"auc_medio": "{:.5f}", "desvio_padrao": "{:.5f}"})

best_params_: {'classificador__C': 0.01}
best_score_: 0.6578


,C,auc_medio,desvio_padrao,ranking
0,0.010000,0.65778,0.00954,1
1,0.100000,0.65748,0.00973,2
2,1.000000,0.65744,0.00973,3
3,10.000000,0.65743,0.00974,5
4,100.000000,0.65743,0.00974,4


O melhor valor encontrado foi C = 0.01, com ROC AUC médio de 0,6578 nas 5 dobras de validação.

Pela tabela, a diferença entre os cinco valores de C é pequena, todos ficam entre 0,6574 e 0,6578, enquanto o desvio padrão entre as dobras é de cerca de 0,0095. Dessa forma, o C = 0.01 vence por pouco, e o modelo é pouco sensível à regularização nessa faixa.

## **EXERCÍCIO 7**

### 7.1 Identifique, no fluxo descrito, exatamente em que ponto ocorre o vazamento de dados (data leakage) e por que a métrica de teste reportada nesse fluxo é otimista demais.

O fluxo descrito pelo estagiário é:

1. Ajustar o `StandardScaler` com o dataset completo `X`
2. Fazer o `train_test_split`
3. Treinar e avaliar o modelo

O vazamento ocorre no passo 1, o `fit` calcula a média e o desvio padrão usando todas as linhas, inclusive as que depois irão para o conjunto de teste. Com isso, o teste deixa de ser um conjunto totalmente desconhecido, pois já influenciou a escala em que os dados são representados.

Por isso, a métrica de teste desse fluxo é otimista**, já que ela mede o desempenho em dados que participaram do pré-processamento, e não em dados realmente novos, como os que chegam em produção.

### 7.2 Reescreva o fluxo correto, na ordem certa, usando train_test_split estratificado antes de qualquer fit de transformação, conforme praticado no Exercício 3.

In [16]:
X_tr, X_te, y_tr, y_te = dividir_treino_teste(X, y)

pipeline_correto = clone(pipeline).fit(X_tr, y_tr)
auc_correto = roc_auc_score(y_te, pipeline_correto.predict_proba(X_te)[:, 1])

print(f"ROC AUC no teste (fluxo correto): {auc_correto:.6f}")

ROC AUC no teste (fluxo correto): 0.664151


No fluxo correto, a ordem é:

1. Dividir os dados brutos* com `train_test_split` estratificado
2. Ajustar o Pipeline (pré-processamento e classificador) apenas com o treino 3. Avaliar no teste, que só é transformado com o que foi aprendido no treino

### 7.3 Documente a auditoria, incluindo o fluxo incorreto, o fluxo corrigido e a justificativa técnica da diferença entre eles.

In [17]:
preprocessador_incorreto = clone(preprocessador)
X_transformado = preprocessador_incorreto.fit_transform(X)

X_tr_inc, X_te_inc, y_tr_inc, y_te_inc = dividir_treino_teste(X_transformado, y)

modelo_incorreto = LogisticRegression().fit(X_tr_inc, y_tr_inc)
auc_incorreto = roc_auc_score(y_te_inc, modelo_incorreto.predict_proba(X_te_inc)[:, 1])

print(f"ROC AUC no teste (fluxo incorreto): {auc_incorreto:.6f}")

ROC AUC no teste (fluxo incorreto): 0.664148


**Fluxo incorreto:**

1. Ajustar o pré-processamento (`fit_transform`) com o `X` completo
2. Dividir os dados já transformados em treino e teste
3. Treinar no treino e avaliar no teste

**Fluxo corrigido:**

1. Dividir os dados brutos com `train_test_split` estratificado (`stratify = y`, `test_size = 0.2`)
2. Ajustar o Pipeline apenas com `X_train` e `y_train`
3. Avaliar com `predict_proba` em `X_test`, que só é transformado com o que foi aprendido no treino

**Evidência observada:** as estatísticas aprendidas pelo scaler são diferentes nos dois fluxos. A média de `uso_medio_semanal_horas` muda cerca de 0,014 horas, a de `tempo_assinatura_meses` cerca de 0,012 meses, e o desvio padrão muda até cerca de 0,008. Essas diferenças existem justamente porque, no fluxo incorreto, as linhas de teste entram no cálculo. Já a métrica de teste ficou praticamente idêntica (ROC AUC de 0,664148 no fluxo incorreto e 0,664151 no corrigido).

**Justificativa técnica:** o vazamento é um problema de processo, o fluxo incorreto permite que o conjunto de teste influencie o ajuste, o que invalida a ideia de que o teste simula dados nunca vistos. Nesta base o efeito na métrica é desprezível, pois o `StandardScaler` vaza só duas estatísticas por coluna, estimadas com 12 mil linhas, e o `LogisticRegression` é pouco sensível a deslocamentos tão pequenos. Isso não torna o fluxo aceitável, o impacto cresce com bases pequenas, com outliers e com etapas que aprendem mais dos dados ou usam o alvo (seleção de variáveis, imputação, PCA, target encoding). Com o fluxo corrigido, a validade da métrica não depende de sorte e reproduz o cenário de produção, em que dados novos nunca participam do `fit`.

## **EXERCÍCIO 8**

### 8.1 Integre em um único script pipeline_final_churn.py as etapas dos Exercícios 3 a 6: divisão estratificada, ColumnTransformer, Pipeline com LogisticRegression e GridSearchCV.

In [18]:
%%writefile pipeline_final_churn.py
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Carregamento dos dados e divisão estratificada em treino e teste
df = pd.read_csv("assinantes_neostream.csv")

colunas_numericas = ["tempo_assinatura_meses", "uso_medio_semanal_horas", "numero_reclamacoes"]
colunas_categoricas = ["plano", "metodo_pagamento", "dispositivo_principal"]

X, y = df[colunas_numericas + colunas_categoricas], df["churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, stratify = y, random_state = 42
)

# Pré-processamento: padronização das numéricas e one-hot das categóricas
preprocessador = ColumnTransformer(
    transformers = [
        ("numericas", StandardScaler(), colunas_numericas),
        ("categoricas", OneHotEncoder(handle_unknown = "ignore"), colunas_categoricas),
    ]
)

# Pipeline: pré-processamento + classificador em um único objeto
pipeline = Pipeline(
    steps = [
        ("preprocessamento", preprocessador),
        ("classificador", LogisticRegression()),
    ]
)

# Busca em grade do hiperparâmetro C, com validação cruzada sobre o Pipeline inteiro
grade_c = {"classificador__C": [0.01, 0.1, 1, 10, 100]}
busca = GridSearchCV(pipeline, param_grid = grade_c, cv = 5, scoring = "roc_auc")
busca.fit(X_train, y_train)

Writing pipeline_final_churn.py


O script `pipeline_final_churn.py` reúne, em um único arquivo, as etapas dos Exercícios 3 a 6, incluindo a divisão estratificada, o `ColumnTransformer`, o `Pipeline` com `LogisticRegression` e o `GridSearchCV`. Ele é independente do notebook, já que carrega o CSV diretamente, e não tem nenhuma transformação manual fora do Pipeline. As colunas de entrada são selecionadas de forma explícita, deixando `cliente_id` e `ltv_alto` de fora do modelo.

### 8.2 Ajuste o Pipeline final com os melhores hiperparâmetros encontrados e treine-o sobre todo o X_train.

In [19]:
%%writefile -a pipeline_final_churn.py

# Pipeline final: melhores hiperparâmetros, treinado sobre todo o X_train
pipeline_final = pipeline.set_params(**busca.best_params_)
pipeline_final.fit(X_train, y_train)

print(f"Melhores hiperparâmetros: {busca.best_params_}")
print(f"ROC AUC médio na validação cruzada: {busca.best_score_:.4f}")
print(f"ROC AUC no teste: {roc_auc_score(y_test, pipeline_final.predict_proba(X_test)[:, 1]):.4f}")

Appending to pipeline_final_churn.py


In [20]:
%run pipeline_final_churn.py

Melhores hiperparâmetros: {'classificador__C': 0.01}
ROC AUC médio na validação cruzada: 0.6578
ROC AUC no teste: 0.6640


Depois da busca, o Pipeline final recebe os melhores hiperparâmetros (`set_params(**busca.best_params_)`) e é treinado sobre todo o `X_train`. Só então o teste é usado, uma única vez, para medir o modelo final.

Ao executar o script, a busca escolheu `C = 0.01`, com ROC AUC médio de 0,6578 na validação cruzada, os mesmos valores do Exercício 6. No teste, o Pipeline final obteve ROC AUC de 0,6640. Validação e teste estão próximos, sem sinal de que o modelo tenha se ajustado em excesso aos dados de validação.

### 8.3 Inclua no script uma seção de comentários explicando cada decisão de design (por que estratificar, por que usar ColumnTransformer, por que Pipeline em vez de passos manuais).

In [21]:
%%writefile -a pipeline_final_churn.py

# =====================================================================
# DECISÕES DE DESIGN
# =====================================================================
# Por que estratificar: o churn é a classe minoritária (cerca de 18%). Com
# stratify = y, treino e teste mantêm essa proporção, e a avaliação não depende
# de uma divisão desfavorável.
#
# Por que selecionar as colunas explicitamente: apenas as seis features do modelo
# entram em X. cliente_id (identificador) e ltv_alto (alvo de outro problema)
# ficam de fora.
#
# Por que usar ColumnTransformer: colunas numéricas e categóricas exigem
# tratamentos diferentes (StandardScaler e OneHotEncoder), e o ColumnTransformer
# reúne os dois em um único objeto. O handle_unknown = "ignore" evita erro com
# categorias inéditas em produção.
#
# Por que Pipeline em vez de passos manuais: pré-processamento e modelo formam um
# só objeto, ajustado de uma vez e apenas com X_train. O mesmo objeto é usado em
# produção, sem repetir transformações à mão, o que evita data leakage e
# inconsistência entre treino e produção.
#
# Por que GridSearchCV sobre o Pipeline: o pré-processamento é reajustado em cada
# dobra da validação cruzada, sem vazamento. A grade é pequena e logarítmica para
# limitar o custo computacional (5 valores x 5 dobras = 25 treinos).
#
# Por que roc_auc: com cerca de 18% de churn, a acurácia é enganosa (prever sempre
# "não cancela" já acerta cerca de 82%). A AUC mede o quanto o modelo ordena bem
# os assinantes por risco.
#
# Por que usar o teste só no final: o conjunto de teste é usado uma única vez,
# para medir o modelo final, e não participa de nenhuma decisão.

Appending to pipeline_final_churn.py


A seção de comentários foi adicionada ao final do `pipeline_final_churn.py` e explica cada decisão de design, incluindo estratificar a divisão, selecionar as colunas explicitamente, usar o `ColumnTransformer`, usar o `Pipeline` em vez de passos manuais, aplicar o `GridSearchCV` sobre o Pipeline inteiro com a métrica `roc_auc` e reservar o teste apenas para a avaliação final.

### 8.4 Escreva um parágrafo final explicando por que essa organização específica é mais segura para produção do que transformar os dados manualmente antes de treinar.

Essa organização é mais segura para produção do que transformar os dados manualmente antes de treinar porque tudo o que é aprendido no treino, médias e desvios do scaler, categorias do encoder e coeficientes do modelo, fica guardado em um único objeto, ajustado de uma só vez e apenas com `X_train`. Isso impede que informação de teste ou de validação vaze para o pré-processamento, já que o `GridSearchCV` reajusta o Pipeline inteiro em cada dobra, e garante que os mesmos passos, na mesma ordem e com os mesmos parâmetros, sejam aplicados a qualquer dado novo com uma única chamada a `predict`. Com passos manuais, cada novo lote exigiria repetir as transformações à mão, o que abre espaço para esquecer uma etapa, trocar a ordem das colunas ou recalcular estatísticas com os dados de produção, gerando previsões silenciosamente erradas. Além disso, a estratificação preserva a proporção de churn na avaliação, o `handle_unknown = "ignore"` evita falhas com categorias inéditas, e o script independente e comentado facilita a revisão por outro engenheiro antes do deploy.